# 🎓 Student Placement Prediction — EDA & Modelling
**AICTE ML Internship Project**

> ⚠️ This notebook expects to be run from inside the `notebooks/` folder (its default location in the project). If you move it elsewhere, update the `DATA_PATH` in the next cell.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
%matplotlib inline

# Portable path: works whether the notebook is run from notebooks/ (default)
# or from the project root, by checking both locations.
_candidates = [
    os.path.join('..', 'dataset', 'student_placement.csv'),
    os.path.join('dataset', 'student_placement.csv'),
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), _candidates[0])
print('Using dataset path:', os.path.abspath(DATA_PATH))

## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## 2. Data Info & Null Check

In [ ]:
print(df.info())
print('\nNull values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget distribution:\n', df['Placed'].value_counts())

## 3. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 4. Exploratory Data Analysis (EDA)

### 4.1 Placement Count (Target Balance)

In [ ]:
plt.figure(figsize=(5, 4))
ax = sns.countplot(data=df, x='Placed', hue='Placed', palette=['#F72585', '#4CC9F0'], legend=False)
ax.set_xticklabels(['Not Placed', 'Placed'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.title('Placement Distribution')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

### 4.2 Feature Distributions by Placement Status

In [ ]:
num_feats = ['CGPA', 'Attendance', 'AptitudeScore',
             'CommunicationSkills', 'TechnicalSkills', 'SoftSkills']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, feat in zip(axes, num_feats):
    sns.histplot(data=df, x=feat, hue='Placed', bins=20, alpha=0.6,
                 palette=['#F72585', '#4CC9F0'], ax=ax, element='step')
    ax.set_title(feat)
plt.suptitle('Feature Distributions by Placement Status', y=1.02)
plt.tight_layout()
plt.show()

### 4.3 Boxplots — Spotting Outliers & Separation by Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, feat in zip(axes, ['CGPA', 'AptitudeScore', 'TechnicalSkills']):
    sns.boxplot(data=df, x='Placed', y=feat, hue='Placed',
                palette=['#F72585', '#4CC9F0'], ax=ax, legend=False)
    ax.set_xticklabels(['Not Placed', 'Placed'])
    ax.set_title(f'{feat} vs Placement')
plt.tight_layout()
plt.show()

### 4.4 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

### 4.5 Pairplot — Multi-feature Relationships

In [ ]:
sample_feats = ['CGPA', 'AptitudeScore', 'TechnicalSkills', 'CommunicationSkills', 'Placed']
sns.pairplot(df[sample_feats], hue='Placed', palette=['#F72585', '#4CC9F0'],
             diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
plt.suptitle('Pairwise Relationships', y=1.02)
plt.show()

## 5. Train / Test Split & Model Training

In [ ]:
X = df.drop('Placed', axis=1)
y = df['Placed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

trained = {}
for name, m in models.items():
    Xtr = X_train_s if name == 'Logistic Regression' else X_train
    Xte = X_test_s  if name == 'Logistic Regression' else X_test
    m.fit(Xtr, y_train)
    y_pred = m.predict(Xte)
    acc = accuracy_score(y_test, y_pred)
    trained[name] = {'model': m, 'y_pred': y_pred, 'accuracy': acc}
    print(f'{name:30s}  Accuracy: {acc:.4f}')

## 6. Detailed Evaluation

In [ ]:
for name, res in trained.items():
    print(f'\n{"="*55}\n{name}\n{"="*55}')
    print(classification_report(y_test, res['y_pred'], target_names=['Not Placed', 'Placed']))

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Not Placed', 'Placed'], yticklabels=['Not Placed', 'Placed'])
    ax.set_title(name)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
rf = trained['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 6))
importances.plot.barh(color='#58A6FF')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Final Model Comparison

In [ ]:
summary = pd.DataFrame({
    'Model': list(trained.keys()),
    'Accuracy': [round(v['accuracy']*100, 2) for v in trained.values()]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)
summary